## Reformulación de la Pregunta Maestra Final

Tras validar la topología de la información mediante algoritmos de densidad (DBSCAN) y purgar las anomalías (Estudio de Ruido), el sistema ha destilado el conocimiento médico en "Macro-Tópicos" puros. Sin embargo, las preguntas que componen cada Macro-Tópico presentan variaciones sintácticas. Para alcanzar la excelencia en la usabilidad clínica, es necesario fusionar estas variantes en una única Categoría Universal.

Los objetivos de este notebook son: alimentar al comité de 5 Modelos de Lenguaje (LLMs) con todas las preguntas pertenecientes a un mismo Macro-Tópico para que realicen una abstracción semántica; obligar a cada modelo a redactar una nueva y definitiva "Pregunta Maestra" que resuma perfectamente la intención de sus predecesoras sin perder rigor médico; y someter estas nuevas candidatas a una última criba matemática para seleccionar algorítmicamente la Pregunta Maestra Final de cada sección, conformando así el *Gold Standard* definitivo del proyecto.

### Síntesis Generativa
Para generar un índice definitivo libre de redundancias, se recurre de nuevo a la técnica de **Síntesis Generativa mediante LLMs**. 

**Metodología de *Prompt Engineering*:**
Se diseña un *prompt* con restricciones estrictas (Zero-Shot) que inyecta todas las variantes de un clúster como contexto. Se instruye al comité de 5 modelos (Qwen3, DeepSeek-R1, Gemma3, GPT-OSS, Nemotron) bajo los siguientes parámetros críticos:
1. **Role-Play Clínico:** Actuar como paciente (uso de la primera persona).
2. **Restricción de Formato:** Prohibición absoluta de preámbulos o explicaciones (extracción de cadena pura).
3. **Temperatura de Inferencia (`0.3`):** Se reduce la "creatividad" térmica de las redes neuronales para favorecer un comportamiento determinista, garantizando que la síntesis sea conservadora y no invente conceptos ajenos al contexto proporcionado.

In [ ]:
import pandas as pd
import requests

URL_OLLAMA = "https://wiig.dia.fi.upm.es/ollama/v1/chat/completions"
MODELOS = [
    'qwen3:30b', 
    'deepseek-r1:32b', 
    'gemma3:27b', 
    'gpt-oss:120b', 
    'nemotron-3-nano:30b'
]

df_clusters = pd.read_csv('clusters_preguntas_final.csv') 

def generar_pregunta_candidata(modelo, lista_preguntas):
    texto_preguntas = "\n".join([f"- {p}" for p in lista_preguntas])
    
    prompt = f"""Actúa como un paciente que está leyendo un prospecto médico.
Tienes las siguientes dudas similares que te han surgido:
{texto_preguntas}

Redacta una ÚNICA pregunta clara, directa y sencilla que englobe todas estas dudas.
Reglas estrictas:
1. Usa lenguaje cotidiano, en primera persona (ej. "¿Puedo...?", "¿Qué pasa si...?").
2. No uses jerga médica compleja a menos que esté en las preguntas originales.
3. Devuelve ÚNICAMENTE la pregunta final, sin comillas, sin introducciones y sin texto extra.
"""
    
    payload = {
        "model": modelo,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.3 # Temperatura baja para evitar respuestas demasiado creativas o largas
    }
    
    try:
        response = requests.post(URL_OLLAMA, json=payload, timeout=150)
        if response.status_code == 200:
            contenido = response.json()['choices'][0]['message']['content'].strip()
            
            # Limpieza específica para DeepSeek
            if "</think>" in contenido:
                contenido = contenido.split("</think>")[-1].strip()
                
            return contenido.replace('"', '').replace("'", "")
        else:
            return f"Error HTTP {response.status_code}"
    except Exception as e:
        return f"Error: {str(e)}"


print("Iniciando generación de candidatas por cluster...")
resultados_candidatas = []

# Filtramos para procesar solo los clusters válidos (ignoramos el ruido -1)
clusters_validos = sorted(df_clusters[df_clusters['Cluster'] != -1]['Cluster'].unique())

for c_id in clusters_validos:
    preguntas_orig = df_clusters[df_clusters['Cluster'] == c_id]['Pregunta'].tolist()
    print(f"\n--- Cluster {c_id} ({len(preguntas_orig)} preguntas) ---")
    
    fila = {
        'Cluster_ID': c_id,
        'Preguntas_Originales': " | ".join(preguntas_orig)
    }
    
    # Preguntamos a cada modelo
    for mod in MODELOS:
        nombre_corto = mod.split(":")[0]
        print(f"  > Consultando a {nombre_corto}...")
        candidata = generar_pregunta_candidata(mod, preguntas_orig)
        fila[f'Candidata_{nombre_corto}'] = candidata
        
    resultados_candidatas.append(fila)


df_candidatas = pd.DataFrame(resultados_candidatas)
df_candidatas.to_csv("candidatas_maestras_por_modelo.csv", index=False, encoding='utf-8-sig')



print(df_candidatas.head(2))

Iniciando generación de candidatas por cluster...

--- Cluster 0 (4 preguntas) ---
  > Consultando a qwen3...
  > Consultando a deepseek-r1...
  > Consultando a gemma3...
  > Consultando a gpt-oss...
  > Consultando a nemotron-3-nano...

--- Cluster 1 (5 preguntas) ---
  > Consultando a qwen3...
  > Consultando a deepseek-r1...
  > Consultando a gemma3...
  > Consultando a gpt-oss...
  > Consultando a nemotron-3-nano...

--- Cluster 2 (14 preguntas) ---
  > Consultando a qwen3...
  > Consultando a deepseek-r1...
  > Consultando a gemma3...
  > Consultando a gpt-oss...
  > Consultando a nemotron-3-nano...

--- Cluster 3 (5 preguntas) ---
  > Consultando a qwen3...
  > Consultando a deepseek-r1...
  > Consultando a gemma3...
  > Consultando a gpt-oss...
  > Consultando a nemotron-3-nano...

--- Cluster 4 (4 preguntas) ---
  > Consultando a qwen3...
  > Consultando a deepseek-r1...
  > Consultando a gemma3...
  > Consultando a gpt-oss...
  > Consultando a nemotron-3-nano...

--- Cluster 5

In [6]:
df_candidatas = pd.read_csv("candidatas_maestras_por_modelo.csv")
df_candidatas

,Cluster_ID,Preguntas_Originales,Candidata_qwen3,Candidata_deepseek-r1,Candidata_gemma3,Candidata_gpt-oss,Candidata_nemotron-3-nano
0,0,¿Puedo tomar el medicamento si estoy embarazad...,¿Puedo tomar este medicamento si estoy embaraz...,¿Puedo tomar este medicamento si estoy embaraz...,¿Puedo tomar este medicamento si estoy embaraz...,¿Puedo tomar este medicamento si estoy embaraz...,¿Puedo tomar este medicamento si estoy embaraz...
1,1,¿Qué debo hacer si olvido o pierdo una dosis d...,¿Qué hago si olvido o pierdo una dosis?,¿Qué debo hacer si olvido una dosis?,¿Qué hago si me olvido de tomar una dosis?,¿Qué hago si olvido o pierdo una dosis de mi m...,¿Qué debo hacer si olvido o pierdo una dosis d...
2,2,"Efectos secundarios, composición, cómo se pres...","¿Qué efectos secundarios debo vigilar, como co...",¿Qué efectos secundarios puede causar este med...,"Antes de empezar a tomar este medicamento, nec...","¿Qué debo saber sobre este medicamento, como s...",¿Puedo tomar este medicamento y qué efectos se...
3,3,Is it safe to drive or operate machinery while...,Error HTTP 500,¿Puedo conducir o usar maquinaria mientras tom...,¿Puedo conducir o usar maquinaria mientras tom...,¿Puedo conducir o usar maquinaria mientras tom...,¿Puedo conducir o usar maquinaria mientras tom...
4,4,¿Puedo tomar este medicamento? | ¿Puedo tomar ...,Error HTTP 500,¿Puedo tomar este medicamento junto con los de...,¿Hay algún problema si tomo este medicamento c...,¿Puedo tomar este medicamento si ya estoy usan...,¿Puedo tomar este medicamento si ya estoy toma...
5,5,¿Qué componentes (principio activo y excipient...,Error HTTP 500,¿Qué componentes (principio activo y excipient...,"¿Qué lleva este medicamento exactamente, tanto...",¿Qué ingredientes y componentes lleva este med...,"¿Puedo saber qué componentes, principio activo..."
6,6,¿Cuándo se revisó por última vez este prospect...,Error HTTP 500,¿Cuándo se revisó o actualizó por última vez l...,¿Podrías decirme cuándo se actualizó por últim...,¿Cuándo fue la última vez que se revisó o actu...,¿Puedo saber cuándo fue la última revisión o a...
7,7,¿Cuál es la dosis y la frecuencia de administr...,Error HTTP 500,¿Cuál es la dosis correcta y con qué frecuenci...,¿Cómo y con qué frecuencia tengo que tomar est...,¿Cuál es la dosis correcta de este medicamento...,¿Qué dosis debo tomar y cada cuánto tiempo?


### Selección Algorítmica
Para evitar cualquier sesgo humano en la elección de la Pregunta Maestra definitiva, se diseña un proceso de evaluación matemática automatizada. Las 5 preguntas candidatas de cada Macro-Tópico compiten entre sí para demostrar cuál encapsula mejor la "verdad semántica" de las dudas originales.

**Métrica de Evaluación (El Juez Matemático):**
1. **Motor Vectorial:** Se reutiliza el modelo `multilingual-e5-large-ft-sts-spanish`, altamente calibrado para tareas de Similitud Textual Semántica (STS) en el idioma objetivo.
2. **Resonancia Intra-Clúster:** En lugar de comparar la candidata contra un centroide teórico, se proyecta la Pregunta Maestra contra **todas y cada una** de las preguntas originales del Macro-Tópico. Se calcula la *Similitud Coseno Promedio* (`np.mean`).
3. **Criterio de Victoria:** Este enfoque penaliza a los LLMs que hayan omitido matices importantes y premia a la IA que haya logrado una generalización perfecta. La candidata con el coseno promedio más alto es coronada como la "ganadora" de su clúster.

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


print("Cargando modelo de embeddings (El Juez)...")
model_emb = SentenceTransformer('mrm8488/multilingual-e5-large-ft-sts-spanish-matryoshka-768-64-5e')

df_candidatas = pd.read_csv("candidatas_maestras_por_modelo.csv")

modelos_nombres = ['qwen3', 'deepseek-r1', 'gemma3', 'gpt-oss', 'nemotron-3-nano']

print("\nIniciando evaluación de similitud semántica...")
resultados_finales = []

for index, row in df_candidatas.iterrows():
    c_id = row['Cluster_ID']
    preguntas_orig = str(row['Preguntas_Originales']).split(" | ")
    
    print(f"\nEvaluando Cluster {c_id} ({len(preguntas_orig)} preguntas originales)...")
    
    # Vectorizamos las originales y sacamos el centroide
    embs_orig = model_emb.encode(preguntas_orig)
    
    mejor_similitud = -1
    modelo_ganador = ""
    pregunta_ganadora = ""
    
    fila_resultado = {
        'Cluster_ID': c_id,
        'Num_Preguntas': len(preguntas_orig)
    }
    
    for mod in modelos_nombres:
        col_name = f'Candidata_{mod}'
        candidata = str(row[col_name])
        
        # Si hubo un error en la celda 1, la similitud es 0
        if candidata == "nan" or "Error" in candidata or candidata == "":
            sim_promedio = 0.0
        else:
            # Vectorizamos la candidata
            emb_cand = model_emb.encode([candidata])
            
            # Calculamos la similitud de la candidata contra TODAS las originales y hacemos la media
            similitudes = cosine_similarity(emb_cand, embs_orig)[0]
            sim_promedio = np.mean(similitudes)
            
        fila_resultado[f'Similitud_{mod}'] = sim_promedio
        fila_resultado[f'Texto_{mod}'] = candidata
        print(f"   - {mod}: {sim_promedio:.4f}")
        
        # Actualizamos el ganador
        if sim_promedio > mejor_similitud:
            mejor_similitud = sim_promedio
            modelo_ganador = mod
            pregunta_ganadora = candidata
            
    # Guardamos al ganador de este cluster
    fila_resultado['Modelo_Ganador'] = modelo_ganador
    fila_resultado['Similitud_Ganadora'] = mejor_similitud
    fila_resultado['Pregunta_Maestra_Final'] = pregunta_ganadora
    
    resultados_finales.append(fila_resultado)
    print(f"GANADOR: {modelo_ganador} -> '{pregunta_ganadora}'")


df_resultados = pd.DataFrame(resultados_finales)


cols = ['Cluster_ID', 'Num_Preguntas', 'Modelo_Ganador', 'Similitud_Ganadora', 'Pregunta_Maestra_Final']
cols.extend([c for c in df_resultados.columns if c not in cols])
df_resultados = df_resultados[cols]

df_resultados.to_csv("seleccion_pregunta_maestra_final.csv", index=False, encoding='utf-8-sig')

print("\nRecuento de victorias por modelo:")
print(df_resultados['Modelo_Ganador'].value_counts())

Cargando modelo de embeddings (El Juez)...


Loading weights: 100%|██████████| 391/391 [00:01<00:00, 209.74it/s, Materializing param=pooler.dense.weight]                               



Iniciando evaluación de similitud semántica...

Evaluando Cluster 0 (4 preguntas originales)...
   - qwen3: 0.9737
   - deepseek-r1: 0.9737
   - gemma3: 0.9383
   - gpt-oss: 0.9737
   - nemotron-3-nano: 0.9737
GANADOR: qwen3 -> '¿Puedo tomar este medicamento si estoy embarazada, planeo quedar embarazada o estoy amamantando?'

Evaluando Cluster 1 (5 preguntas originales)...
   - qwen3: 0.9573
   - deepseek-r1: 0.9685
   - gemma3: 0.9715
   - gpt-oss: 0.9623
   - nemotron-3-nano: 0.9623
GANADOR: gemma3 -> '¿Qué hago si me olvido de tomar una dosis?'

Evaluando Cluster 2 (14 preguntas originales)...
   - qwen3: 0.8307
   - deepseek-r1: 0.8493
   - gemma3: 0.8407
   - gpt-oss: 0.8286
   - nemotron-3-nano: 0.8331
GANADOR: deepseek-r1 -> '¿Qué efectos secundarios puede causar este medicamento, cómo se toma, qué riesgos tiene especialmente con los ojos o coágulos sanguíneos, y cuándo debo buscar ayuda médica?'

Evaluando Cluster 3 (5 preguntas originales)...
   - qwen3: 0.0000
   - deepseek-

### Análisis del Torneo Semántico
Los resultados del escrutinio arrojan tres conclusiones que confirman el uso de la arquitectura *Ensemble* (Comité de Modelos) diseñada para este proyecto:

**1. Especialización Latente (Empate Técnico):**
Ningún modelo ha demostrado supremacía absoluta, resultando en un empate en cabeza entre **DeepSeek-R1 (3 victorias)** y **GPT-OSS (3 victorias)**. 
* El modelo de OpenAI (`gpt-oss`) confirma su dominio en tareas de síntesis burocrática y descriptiva (Interacciones, Componentes, Fechas de Revisión).
* La arquitectura de razonamiento de `deepseek-r1` se impone en la síntesis de instrucciones clínicas densas. Es especialmente notable su victoria en el **Cluster 2**, donde logró comprimir 14 preguntas originales sobre efectos adversos graves en una única Pregunta Maestra manteniendo una similitud promedio de `0.8493`.

**2. Tolerancia a Fallos en Producción (Resiliencia del Sistema):**
El *log* de ejecución documenta una caída del servidor durante la inferencia del modelo `qwen3` (Error HTTP 500 en los clústeres 3 al 7). En una arquitectura tradicional (Single-LLM), esto habría provocado un fallo catastrófico del *pipeline*. Sin embargo, gracias al diseño del comité y a la penalización matemática del "Juez" (`0.0000` de similitud para respuestas nulas), el sistema absorbió el fallo y seleccionó la mejor alternativa entre los modelos supervivientes.

**3. Calidad de la Síntesis Clínica:**
Las Preguntas Maestras ganadoras alcanzan similitudes promedio superiores a `0.90` (llegando a `0.97` en posología y embarazo). Esto certifica que el tono empático introducido por el *prompt* ("Actúa como un paciente") no ha degradado en absoluto la fidelidad médica de la información.